BreastMNIST Dataset for Classifying Breast masses as either Benign or Malignant from breast ultrasound images  using a **Convolutional Neural Network** (CNN)

In [ ]:
%pip install medmnist
%pip install torch torchvision torchaudio
%pip install transformers datasets

In [ ]:
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_auc_score
import matplotlib.pyplot as plt
from medmnist import BreastMNIST
from medmnist import INFO
import numpy as np
import copy

In [ ]:
# BreastMNIST Information
info = INFO['breastmnist']['label']
print(info)

In [ ]:
# Available accelerator
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"The available device is: {device}")

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),  # Converts image to a PyTorch tensor (C×H×W)
    transforms.Normalize(mean=[0.5], std=[0.5]) # Scale pixel values from [0,1] → [-1,1]
])

# Validation/test transform
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

train_data = BreastMNIST(split='train', download=True, transform=transform)
test_data = BreastMNIST(split='test', download=True, transform=eval_transform)
val_data =BreastMNIST(split = 'val', download=True, transform=eval_transform)

print(f"Number of training samples: {len(train_data)}")
print(f"Number of testing samples: {len(test_data)}")
print(f"Number of validation samples: {len(val_data)}")
print(f"Total number of samples: {len(train_data) + len(test_data) + len(val_data)}")

In [ ]:
# Load dataset
train_set = DataLoader(train_data, batch_size = 32, shuffle = True)
test_set = DataLoader(test_data, batch_size = 64, shuffle = False )
val_set = DataLoader(val_data, batch_size = 64, shuffle = False)

In [ ]:
# Show a few training samples (after augmentation)
images, labels = next(iter(train_set))            # iterate through the batches in trainset
print("Batch shape:", images.shape)               # the shape
fig, axes = plt.subplots(1, 6, figsize=(10, 3))
for i, ax in enumerate(axes):
    ax.imshow(images[i].squeeze(), cmap="gray")       # .squeez remove channel since pyplot can display (28 x 28)
    ax.set_title("malignant" if labels[i].item() == 0 else "normal/benign", fontsize=8)
    ax.axis("off")
plt.show()

In [ ]:
# CNN Architecture
class BreastMNIST_CNN(nn.Module):
  def __init__(self):
    super().__init__()

    # first convolution layer
    self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
    self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

    # Second convolution layer
    self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
    self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

    # Third convolution layer
    self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
    self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

    # Activation function
    self.relu = nn.ReLU()

    # dropout to reduce overfitting
    self.dropout = nn.Dropout(0.5)

    # Fully connected layers
    self.fc1 = nn.Linear(128 * 3 * 3, 512)
    self.fc2 = nn.Linear(512, 2)


  # Pass through an activation function
  def forward(self, x):
    x = self.pool1(self.relu(self.conv1(x)))
    x = self.pool2(self.relu(self.conv2(x)))
    x = self.pool3(self.relu(self.conv3(x)))

    # Flatten
    x = x.view(x.size(0), -1)

    x = self.dropout(self.relu(self.fc1(x)))   # activation between the FC layers
    x = self.fc2(x)
    return x

In [ ]:
# Training parameters
batch_size = 32
num_epochs = 50
learning_rate = 0.001

model = BreastMNIST_CNN().to(device)      # initializing model

# Class weights: label 0 (malignant) is the minority class, so it gets a bigger weight
counts = np.bincount(train_data.labels.ravel())
class_weights = torch.tensor(counts.sum() / (2.0 * counts), dtype=torch.float32).to(device)
print("Class counts:", counts, "| weights:", class_weights.cpu().numpy())

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)

In [ ]:
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

best_val_loss = float("inf")
best_state = None
best_epoch = 0


for epoch in range(num_epochs):
  model.train()                 # training mode
  running_loss = 0
  correct = 0
  total = 0

  for images, labels in train_set:
    images = images.to(device)
    labels = labels.squeeze(1).long().to(device)

    # Reset gradients
    optimizer.zero_grad()

     # Forward pass
    outputs = model(images)

    # Compute loss
    loss = criterion(outputs, labels)

    # Back propagation
    loss.backward()

    # Update weights
    optimizer.step()

    running_loss += loss.item()

    _, predicted = torch.max(outputs, dim=1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()

  train_losses.append(running_loss / len(train_set))
  train_accuracies.append(correct / total * 100)

  # disables dropout, validation mode
  model.eval()
  val_loss = 0
  val_correct = 0
  val_total = 0

  with torch.no_grad():
    for images, labels in val_set:
      images = images.to(device)
      labels = labels.squeeze(1).long().to(device)

      outputs = model(images)
      loss = criterion(outputs, labels)

      val_loss += loss.item()

      _, predicted = torch.max(outputs, dim=1)
      val_total += labels.size(0)
      val_correct += (predicted == labels).sum().item()

  # run ONCE per epoch, after all validation batches
  val_losses.append(val_loss / len(val_set))
  val_accuracies.append(val_correct / val_total * 100)

  # keep the best model
  if val_losses[-1] < best_val_loss:
    best_val_loss = val_losses[-1]
    best_epoch = epoch + 1
    best_state = copy.deepcopy(model.state_dict())

  print(f"Epoch {epoch+1}/{num_epochs} | "
          f"Train Loss: {train_losses[-1]:.4f} | Val Loss: {val_losses[-1]:.4f} | "
          f"Train Acc: {train_accuracies[-1]:.2f}% | Val Acc: {val_accuracies[-1]:.2f}%")

model.load_state_dict(best_state)              # restore the best epoch
print(f"\nBest epoch: {best_epoch} (val loss {best_val_loss:.4f})")





In [ ]:
# Loss graph
plt.figure(figsize=(6,4))
plt.plot(range(1, num_epochs + 1), train_losses, label="Train Loss")
plt.plot(range(1, num_epochs + 1), val_losses, label="Validation Loss")
plt.axvline(best_epoch, color="gray", linestyle="--", label=f"Best epoch ({best_epoch})")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

# Accuracy graph
plt.figure(figsize=(6,4))
plt.plot(range(1, num_epochs + 1), train_accuracies, label="Train Accuracy")
plt.plot(range(1, num_epochs + 1), val_accuracies, label="Validation Accuracy")
plt.axvline(best_epoch, color="gray", linestyle="--", label=f"Best epoch ({best_epoch})")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Testing model on an unseen data
model.load_state_dict(best_state)
model.eval()

probs, ys = [], []
correct, total = 0, 0
with torch.no_grad():
    for images, labels in test_set:
        outputs = model(images.to(device))
        preds = outputs.argmax(dim=1).cpu()
        labels = labels.squeeze(1).long()
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        probs += F.softmax(outputs, dim=1)[:, 0].cpu().tolist()   # P(malignant), using outputs directly instead of running model() again
        ys += labels.tolist()


print(f"Test accuracy: {correct / total:.2%}")
print(f"AUC: {roc_auc_score(np.array(ys) == 0, probs):.3f}")

In [ ]:
# computing performance metrics with confusion matrics from test dataset
model.load_state_dict(best_state)
model.eval()

y_true, y_pred = [], []
with torch.no_grad():
    for images, labels in test_set:
        y_pred += model(images.to(device)).argmax(1).cpu().tolist()
        y_true += labels.squeeze(1).tolist()

cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["malignant", "benign"]).plot(cmap="Blues")
plt.show()


TP, FN, FP, TN = cm[0, 0], cm[0, 1], cm[1, 0], cm[1, 1]
print(f"Accuracy: {(TP+TN)/cm.sum():.2%}")
print(f"Sensitivity: {TP/(TP+FN):.2%}")
print(f"Specificity: {TN/(TN+FP):.2%}")